#### Determinizing and Minimizing Finite State Automata

Consider the following declarations from the course notes:

In [ ]:
class fset(frozenset):
    def __repr__(self):
        return '{' + ', '.join(str(e) for e in self) + '}'

In [ ]:
def wrap(a):
    import textwrap
    return  '\\n'.join(textwrap.wrap(str(a), width=12))

TransFunc = dict[str, dict[str, set[str]]]

class FiniteStateAutomaton:
    Σ: set[str] # set of symbols
    Q: set[str] # set of states
    I: set[str] # I ⊆ Q, the initial states,
    δ: TransFunc # representing Q ↛ Σ ↛ 𝒫Q, the transition function
    F: set[str] # F ⊆ Q, the finite states
    vars = () # for reduced FSAs, the names of the original variables
    def __init__(self, Σ, Q, I, δ, F):
        self.Σ, self.Q, self.I, self.δ, self.F = Σ, fset(Q), fset(I), δ, fset(F)
    def draw(self, trace = None):
        from graphviz import Digraph
        dot = Digraph(graph_attr={'rankdir':'LR'},
                      node_attr={'fontsize': '10', 'fontname': 'Noto Sans', 'margin': '0', 'width': '0.25'}, # 'nodesep': '0.75', 'ranksep': '0.75'
                      edge_attr={'fontsize': '10', 'fontname': 'Noto Sans', 'arrowsize': '0.5'}) # 'weight': '5.0' # create a directed graph
        for q in self.I:
            dot.node('_' + str(q), label = '', shape = 'none', height = ".0", width = ".0") 
            dot.node(wrap(q), shape = 'circle')
            dot.edge('_' + str(q), wrap(q), len = '.1')
        P = self.I | self.F
        for q in self.δ:
            P = P | {q}
            for a in self.δ[q]:
                dot.node(wrap(q), shape = 'circle')
                for r in self.δ[q][a]:
                    dot.node(wrap(r), shape = 'circle')
                    dot.edge(wrap(q), wrap(r), label = str(a))
                    P = P | {r}
        for q in self.F: dot.node(wrap(q), shape = 'doublecircle')
        for q in self.Q - P: # place all unreachable nodes to the right
            dot.node(wrap(q), shape = 'circle')
            for p in P: dot.edge(wrap(p), wrap(q), style='invis') # , constraint='false'
        if trace:
            xlab = {} # maps states to Graphviz external labels
            for i in range(0, len(trace), 2):
                xlab[trace[i]] = xlab[trace[i]] + ', ' + str(i // 2) if trace[i] in xlab else str(i // 2)
            for q in xlab:
                dot.node(wrap(q), xlabel = '<<font color="royalblue">' + wrap(xlab[q]) + '</font>>')
        return dot
    def writepdf(self, name, trace = None):
        open(name, 'wb').write(self.draw(trace).pipe(format='pdf'))
    def writesvg(self, name, trace = None):
        open(name, 'wb').write(self.draw(trace).pipe(format='svg'))
    def __repr__(self):
        return ' '.join(str(q) for q in self.I) + '\n' + \
               '\n'.join(str(q) + ' ' + str(a) + ' → ' + ', '.join(str(r) for r in self.δ[q][a]) \
                         for q in self.δ for a in self.δ[q] if self.δ[q][a] != set()) + '\n' + \
               ' '.join(str(f) for f in self.F) + '\n'

def parseFSA(fsa: str) -> FiniteStateAutomaton:
    fl = [line for line in fsa.split('\n') if line != '']
    I = set(fl[0].split()) if len(fl) > 0 else set() # second line: initial initial ...
    Σ, Q, δ, F = set(), set(), {}, set()
    for line in fl[1:]: # all subsequent lines
        if '→' in line: # source action → target
            l, r = line.split('→'); p, a, q = l.split()[0], l.split()[1], r.split()[0]
            if p in δ: s = δ[p]; s[a] = s[a] | {q} if a in s else {q}
            else: δ[p] = {a: {q}}
            Σ.add(a); Q.add(p); Q.add(q)
        else: # a line without → is assumed to have the final states
            F = set(line.split()) if len(line) > 0 else set() # final final ...
    return FiniteStateAutomaton(Σ, Q | I | F, I, δ, F)

In [ ]:
def setunion(S: set[set]) -> set:
    return set.union(set(), *S)

def δ̂(δ: TransFunc, P: set[str], a: str) -> set[str]:
    return fset(setunion(δ[p][a] for p in P if p in δ if a in δ[p]))

In [ ]:
def merge(γ: TransFunc, δ: TransFunc) -> TransFunc:
    return {q: γ[q] for q in γ.keys() - δ.keys()} | \
           {q: δ[q] for q in δ.keys() - γ.keys()} | \
           {q: {a: γ[q].get(a, set()) | δ[q].get(a, set())
                for a in γ[q].keys() | δ[q].keys()} for q in γ.keys() & δ.keys()}

In [ ]:
def determinize(A: FiniteStateAutomaton, log = False) -> FiniteStateAutomaton:
    Iʹ = fset({A.I})
    Qʹ, δʹ, V = set(Iʹ), {}, set()
    if log: print(Iʹ, Qʹ, δʹ, V)
    while Qʹ != V:
        qʹ = (Qʹ - V).pop(); V |= {qʹ}
        for a in A.Σ:
            rʹ = δ̂(A.δ, qʹ, a)
            if rʹ != set(): Qʹ |= {rʹ}; δʹ = merge(δʹ, {qʹ: {a: {rʹ}}})
        if log: print(Qʹ, δʹ, V)
    Fʹ = {qʹ for qʹ in Qʹ if qʹ & A.F != set()}
    return FiniteStateAutomaton(A.Σ, Qʹ, Iʹ, δʹ, Fʹ)

In [ ]:
def equiv(A: FiniteStateAutomaton, B: FiniteStateAutomaton, log = False) -> bool:
    W, V = {(A.I, B.I)}, set() # work, visited
    while W != set():
        P, Q = W.pop()
        if (P, Q) not in V:
            if log: print('checking', P, Q)
            if (P & A.F == set()) != (Q & B.F == set()): return False
            for a in A.Σ | B.Σ:
                W |= {(δ̂(A.δ, P, a), δ̂(B.δ, Q, a))}
            V |= {(P, Q)}
    if log: print('equivalent', V)
    return True

In [ ]:
def minimize(A: FiniteStateAutomaton, log = False) -> FiniteStateAutomaton:
    δ = {q: {a: list(A.δ[q][a])[0] for a in A.δ[q]} for q in A.δ}
    if log: print('δ =', δ)
    eq = {(q, r) for q in A.Q for r in A.Q if (q in A.F) == (r in A.F)}
    done = False
    while not done:
        if log: print('eq = ', sorted(eq))
        done = True
        for q in δ:
            for r in δ:
                if (q, r) in eq and ((q in δ) != (r in δ) or (δ[q].keys() != δ[r].keys()) or \
                                        any((δ[q][a], δ[r][a]) not in eq for a in δ[q])):
                    eq -= {(q, r)}; done = False
    prt = {fset({q} | {r for r in A.Q if (q, r) in eq}) for q in A.Q}
    if log: print('prt = ', prt)
    rep = {q: set(qʹ).pop() for qʹ in prt for q in qʹ}
    if log: print('rep = ', rep)
    δʹ = {q: {a: {rep[δ[q][a]]} for a in δ[q]} for q in rep.values() if q in δ}
    Iʹ = {rep[q] for q in A.I}
    Fʹ = {rep[q] for q in A.F}
    return FiniteStateAutomaton(A.Σ, rep.values(), Iʹ, δʹ, Fʹ)

In [ ]:
def rename(A: FiniteStateAutomaton, log = False) -> FiniteStateAutomaton:
    m, c = {}, 0
    W, V = A.I, set()
    while W != set():
        for q in W: m[q] = c; c += 1
        W, V = setunion(δ̂(A.δ, W, a) for a in A.Σ) - W - V, V | W
    if log: print('m = ', m)
    Qʹ = {i for i in range(c)}
    Iʹ = {m[q] for q in A.I}
    δʹ = {m[q]: {a: {m[r] for r in A.δ[q][a]} for a in A.δ[q]} for q in A.δ}
    Fʹ = {m[q] for q in A.F}
    return FiniteStateAutomaton(A.Σ, Qʹ, Iʹ, δʹ, Fʹ)

Consider the two finite state automata: 

In [ ]:
A0 = parseFSA("""
0
0 a → 1
0 b → 2
1 b → 0
2 a → 0
2 a → 3
3 b → 2
0 3
""")
A1 = parseFSA("""
x
x b → z
y b → x
z a → x
x a → y
w a → w
x
""")

1. Draw the finite state diagrams of `A0` and `A1`!

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

2. Determinize `A0` and draw it! Call it `A0det`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

3. Use `equiv` to check if `A0` and `A1` are equivalent! Set the `log` parameter to `True`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

4. What is the relationship between `A0det` and the equivalences of states that `equiv` constructs?

YOUR ANSWER HERE

5. Now minimize `A0det` and draw it! Call it `A0min`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

6. What is the relationship between `A0min` and `A1`?

YOUR ANSWER HERE